# 표준 Notebook 샘플 - Door Defect 탐지 (YOLOv8)

`before_windows_yolo_train.py` (Windows 로컬에서 작성된 원본 학습 코드)를
MLOps 표준 계약(Notebook + Asset Key/Version + Runtime Catalog)으로 전환한 버전.
하이퍼파라미터/증강 설정은 원본과 **동일한 항목을 전부** 유지했다 - 줄어든 건 물리 경로뿐이다.

| 원본의 문제 | 전환 후 |
|---|---|
| `C:\Users\...\dataset` (로컬 절대경로) | `prizm.dataset.download(dataset_key, dataset_version, dest_dir=...)` — 압축 없이 폴더 그대로 |
| `C:\Users\...\yolov8n.pt` (개인 PC 로컬 파일) | `prizm.model.download(base_model_key, base_model_version, dest_dir=...)` |
| `C:\Users\...\runs` (로컬 바탕화면 결과 폴더) | `output_dir` 파라미터 (Airflow/Portal이 실행 시점에 주입) |
| `CUDA_VISIBLE_DEVICES=0` / `device=0` 하드코딩 | Runtime Catalog가 컨테이너 단위로 GPU를 격리 배정 (Notebook은 device만 선언) |
| `Z:\공유\...\label_map.json` 수동 파싱 | 별도 파일 불필요 - 라벨 정의가 Dataset Asset(data.yaml)에 포함되어 배포됨 |
| 로컬 텍스트 파일에 결과 수기 기록 | `mlflow.log_metric()` — 버전별로 조회 가능한 UI |

ML 개발자는 여전히 개발 자체는 자유로운 환경(Windows 등)에서 하되,
**MLOps에 최종 등록하는 실행 자산은 이 표준 Notebook 하나**가 된다.

In [ ]:
# Papermill Parameters
# Portal의 "실행" 화면에서 이 값들을 실행 전에 오버라이드할 수 있다.
output_dir = "./output"
mlflow_tracking_uri = "http://mlflow:5000"
dataset_key = "demo.door_defect"
dataset_version = "v1"
base_model_key = "demo.yolov8n_base"
base_model_version = "v1"
model_key = "demo.door_defect_yolov8"

# Runtime Catalog(GPU 이미지)에서는 device=0 이 해당 컨테이너에 배정된
# 유일한 GPU를 가리키므로 그대로 안전하다. 이 데모(CPU Runtime)는 "cpu".
device = "cpu"

# 학습 하이퍼파라미터 - before_windows_yolo_train.py 와 동일한 항목
epochs = 1
imgsz = 320
batch = 4
workers = 0
seed = 0
optimizer = "SGD"
lr0 = 0.01
lrf = 0.01
momentum = 0.937
weight_decay = 0.0005
warmup_epochs = 3.0
cos_lr = False
patience = 50
close_mosaic = 10
label_smoothing = 0.0
save_period = -1

# 증강(augmentation) 파라미터
hsv_h = 0.015
hsv_s = 0.7
hsv_v = 0.4
degrees = 0.0
translate = 0.1
scale = 0.5
shear = 0.0
perspective = 0.0
flipud = 0.0
fliplr = 0.5
mosaic = 1.0
mixup = 0.0

# 평가 임계값
conf_thres = 0.25
iou_thres = 0.7


### ① MLOps SDK Import

In [ ]:
import os
import yaml
import prizm
import mlflow
from ultralytics import YOLO


### ② Dataset Asset 다운로드
원본의 `C:\Users\jwlee\Desktop\door_project\dataset` 대신, Key+Version으로 받는다.
**압축(zip) 없이** images/labels/data.yaml 폴더 구조 그대로 S3에 저장되어 있고,
`dest_dir`로 지정한 경로에 그대로 복원된다 - 별도 압축 해제 코드가 필요 없다.

In [ ]:
dataset_dir = prizm.dataset.download(dataset_key, dataset_version, dest_dir="./datasets/door_defect")
dataset_dir = os.path.abspath(dataset_dir)
data_yaml = os.path.join(dataset_dir, "data.yaml")

# ultralytics는 yaml의 path를 (파일 위치가 아니라) 자체 DATASETS_DIR
# 설정 기준 상대경로로 해석한다. 방금 실제로 받은 절대경로로 명시적으로
# 고쳐써서 이 위치 불확실성을 제거한다 (압축 여부와는 무관한 별개 단계).
with open(data_yaml) as f:
    _cfg = yaml.safe_load(f)
_cfg["path"] = dataset_dir
with open(data_yaml, "w") as f:
    yaml.safe_dump(_cfg, f, sort_keys=False, allow_unicode=True)

print("dataset ready:", data_yaml)


### ③ Baseline Weight(Model Asset) 다운로드
원본의 `C:\Users\jwlee\Downloads\yolov8n.pt` 대신, Key+Version으로 받는다.

In [ ]:
base_weight_path = prizm.model.download(base_model_key, base_model_version, dest_dir="./models")
model = YOLO(base_weight_path)


### ④ Training
하이퍼파라미터/증강 설정은 원본 스크립트와 동일하게 전부 전달한다 - 줄어든 건 물리 경로뿐이다.

In [ ]:
os.makedirs(output_dir, exist_ok=True)
model.train(
    data=data_yaml,
    epochs=epochs,
    imgsz=imgsz,
    batch=batch,
    workers=workers,
    seed=seed,
    device=device,
    optimizer=optimizer,
    lr0=lr0,
    lrf=lrf,
    momentum=momentum,
    weight_decay=weight_decay,
    warmup_epochs=warmup_epochs,
    cos_lr=cos_lr,
    patience=patience,
    close_mosaic=close_mosaic,
    label_smoothing=label_smoothing,
    save_period=save_period,
    hsv_h=hsv_h,
    hsv_s=hsv_s,
    hsv_v=hsv_v,
    degrees=degrees,
    translate=translate,
    scale=scale,
    shear=shear,
    perspective=perspective,
    flipud=flipud,
    fliplr=fliplr,
    mosaic=mosaic,
    mixup=mixup,
    project=output_dir,
    name="train",
    exist_ok=True,
    verbose=False,
)


### ⑤ Evaluation

In [ ]:
metrics = model.val(data=data_yaml, device=device, workers=workers, conf=conf_thres, iou=iou_thres)
map50 = float(metrics.box.map50)
print(f"mAP50: {map50:.4f}")


### ⑥ 산출물(best.pt) 업로드
원본에서는 `학습 결과를 로컬 텍스트 파일에 수기로 남기고, 메신저로 best.pt 파일을 전달`했다.
여기서는 `prizm.model.upload()`로 AI Hub에 자산으로 등록하고, MLflow에 하이퍼파라미터/메트릭을 남긴다.

In [ ]:
best_weight = os.path.join(output_dir, "train", "weights", "best.pt")

mlflow.set_tracking_uri(mlflow_tracking_uri)
mlflow.set_experiment("mlops-core-demo")

with mlflow.start_run(run_name="yolov8-door-defect-training"):
    mlflow.log_params({
        "base_model": f"{base_model_key}:{base_model_version}",
        "epochs": epochs, "imgsz": imgsz, "batch": batch,
        "optimizer": optimizer, "lr0": lr0, "lrf": lrf,
        "momentum": momentum, "weight_decay": weight_decay,
        "patience": patience, "mosaic": mosaic, "mixup": mixup,
    })
    mlflow.log_metric("mAP50", map50)
    mlflow.log_artifact(best_weight)

    result = prizm.model.upload(model_key, best_weight)
    mlflow.set_tag("mlops_asset_key", result["key"])
    mlflow.set_tag("mlops_asset_version", result["version"])

print("registered:", result)
